# Silver — ecommerce_itens_pedido

Este notebook lê a Bronze Delta `squad1.bronze_ecommerce_itens_pedido`, aplica as 10 regras de qualidade da tabela de itens de pedido, grava a Silver Delta e registra os resultados na tabela compartilhada `squad1.dq_monitoring_logs`.




In [0]:
%run ../utils/utils

## Imports e parâmetros

In [0]:

import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timezone

# Variáveis
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_itens_pedido"
TABELA_DQ = "dq_monitoring_logs"

## Setup e Referências

In [0]:
# 1. Carrega Bronze de Itens
df_bronze = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)

# 2. Anti-Join (Processa apenas novos)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    df_micro_lote = df_bronze.join(df_silver_atual, "id_item_pedido", "left_anti")
else:
    df_micro_lote = df_bronze

# 3. Referências
df_pedidos_ref = ler_delta("bronze", "ecommerce_pedidos", STORAGE_OPTIONS).select("id_pedido", "valor_total")
df_produtos_ref = ler_delta("bronze", "ecommerce_produtos", STORAGE_OPTIONS).select("sku")

## Anti-Join e Referências (Pedidos e Produtos)

In [0]:

df_bronze = ler_delta_spark(CAMINHO_BRONZE)
if "bronze_source_file" not in df_bronze.columns:
    raise Exception("A Bronze precisa conter bronze_source_file.")

df_micro_lote = anti_duplicidade_por_arquivo(df_bronze, CAMINHO_SILVER, "bronze_source_file")
qtd_micro_lote = df_micro_lote.count()
TEM_MICRO_LOTE_NOVO = qtd_micro_lote > 0
print("Registros para processar:", qtd_micro_lote)

df_pedidos_ref = safe_read_delta(CAMINHO_BRONZE_PEDIDOS, "Bronze pedidos")
if df_pedidos_ref is None:
    df_pedidos_ref = spark.createDataFrame([], StructType([StructField("id_pedido", LongType(), True), StructField("valor_total", DoubleType(), True)]))

df_pedidos_ref = df_pedidos_ref.select("id_pedido", F.col("valor_total").cast("double").alias("valor_total_pedido")).dropDuplicates(["id_pedido"])

df_produtos_ref = safe_read_delta(CAMINHO_BRONZE_PRODUTOS, "Bronze produtos")
if df_produtos_ref is None:
    df_produtos_ref = spark.createDataFrame([], StructType([StructField("sku", StringType(), True)]))

df_produtos_ref = df_produtos_ref.select(F.col("sku").cast("string").alias("sku")).dropDuplicates().withColumn("sku_existe", F.lit(True))


## A Muralha de Qualidade (Regras de 1 a 10)

In [0]:
# Janelas
w_pedido = Window.partitionBy("id_pedido")

# Prepara base
df_base = df_micro_lote \
    .join(df_pedidos_ref, "id_pedido", "left") \
    .join(df_produtos_ref, "sku", "left") \
    .withColumn("valor_item_liquido", (F.col("preco_unitario") - F.col("desconto_aplicado")) * F.col("quantidade")) \
    .withColumn("soma_valor_itens", F.sum("valor_item_liquido").over(w_pedido)) \
    .withColumn("qtd_itens_por_pedido", F.count("id_item_pedido").over(w_pedido))

# Regras
df_silver_itens = df_base \
    .withColumn("r1_id_falhou", F.col("id_item_pedido").isNull()) \
    .withColumn("r2_pedido_fk_falhou", F.col("id_pedido").isNull() | F.col("valor_total").isNull()) \
    .withColumn("r3_sku_falhou", F.col("sku").isNull()) \
    .withColumn("r4_qtd_falhou", F.col("quantidade") < 1) \
    .withColumn("r5_preco_falhou", F.col("preco_unitario") <= 0) \
    .withColumn("r6_desconto_maior_preco_falhou", F.col("desconto_aplicado") > F.col("preco_unitario")) \
    .withColumn("r7_consistencia_financeira_falhou", F.abs(F.col("soma_valor_itens") - F.col("valor_total")) > 0.01) \
    .withColumn("r8_desconto_negativo_falhou", F.col("desconto_aplicado") < 0) \
    .withColumn("r9_limite_desconto_falhou", (F.col("desconto_aplicado") / F.col("preco_unitario")) > 0.5) \
    .withColumn("r10_pedido_sem_item_falhou", F.col("qtd_itens_por_pedido") < 1)

## Gravação Blindada e Logs

In [0]:
# Seleção de críticas
flags_criticas = [c for c in df_silver_itens.columns if "falhou" in c and "r7" not in c and "r10" not in c]
condicao_invalida_critica = reduce(lambda a, b: a | b, [F.col(c) for c in flags_criticas])

df_silver_validos = df_silver_itens.filter(~condicao_invalida_critica) \
    .select(*[c for c in df_bronze.columns]) \
    .withColumn("silver_processed_at", F.current_timestamp())

# Gravação (Mesmo padrão das anteriores)
gravar_delta(df_silver_validos, "silver", TABELA_ALVO, STORAGE_OPTIONS, mode="append")

# (Aqui você insere a lógica de gravação de logs em dq_monitoring_logs que já usamos)

## Validação final

In [0]:

print("\n===== Validação final =====")
try:
    print("Silver:", CAMINHO_SILVER)
    df_val_silver = ler_delta_spark(CAMINHO_SILVER)
    print("Registros na Silver:", df_val_silver.count())
    display(df_val_silver.limit(20))
except Exception as e:
    print("Silver ainda não disponível ou vazia:", e)

try:
    df_logs_val = spark.table(TABELA_DQ_LOGS)
    print("Registros totais em dq_monitoring_logs:", df_logs_val.count())
    display(df_logs_val.orderBy(F.col("timestamp_execucao").desc()).limit(30))
except Exception as e:
    print("Não foi possível consultar dq_monitoring_logs:", e)
